# UC-06 – Silver, Gold et qualité

Ce notebook est distinct de l'exploration. Il applique les règles, joint les ventes au référentiel produit, écrit Silver et Gold en Parquet, relit les résultats et dresse le bilan qualité.

Question métier : quel est le chiffre d'affaires net des ventes magasin, par produit, sur le lot initial ?

Les noms et les types viennent de la spécification TPCx-BB 1.7.0, annexe B. Cette annexe ne couvre que les colonnes utilisées par les requêtes ; les autres colonnes du fichier portent le nom de la table complète, après contrôle de la largeur. Les sources ne sont pas modifiées. `mode("overwrite")` ne vise que les répertoires de sortie de ce notebook.

Ordre : lancer toutes les cellules de haut en bas. HDFS doit répondre (`hdfs://localhost:9000`).


In [1]:
import os
import subprocess
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DecimalType, IntegerType, LongType, StringType, StructField, StructType

spark = (
    SparkSession.builder
    .appName("UC06-silver-gold")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

VENTES = "/data/retail_project/data/store_sales/store_sales.dat"
PRODUITS = "/data/retail_project/data/item/item.dat"
SILVER_VENTES = "/retail-lake/silver/uc06/store_sales"
SILVER_PRODUITS = "/retail-lake/silver/uc06/item"
GOLD = "/retail-lake/gold/uc06/ventes_magasin_par_produit"
CSV_COMPARAISON = "/retail-lake/tmp/uc06/ventes_magasin_par_produit_csv"

MONTANT = DecimalType(7, 2)
LECTURE = {"sep": "|", "header": False, "quote": "\u0000", "escape": "\u0000", "mode": "PERMISSIVE"}

SCHEMA_VENTES = StructType([
    StructField("ss_sold_date_sk", LongType()),
    StructField("ss_sold_time_sk", LongType()),
    StructField("ss_item_sk", LongType()),
    StructField("ss_customer_sk", LongType()),
    StructField("ss_cdemo_sk", LongType()),
    StructField("ss_hdemo_sk", LongType()),
    StructField("ss_addr_sk", LongType()),
    StructField("ss_store_sk", LongType()),
    StructField("ss_promo_sk", LongType()),
    StructField("ss_ticket_number", LongType()),
    StructField("ss_quantity", IntegerType()),
    StructField("ss_wholesale_cost", MONTANT),
    StructField("ss_list_price", MONTANT),
    StructField("ss_sales_price", MONTANT),
    StructField("ss_ext_discount_amt", MONTANT),
    StructField("ss_ext_sales_price", MONTANT),
    StructField("ss_ext_wholesale_cost", MONTANT),
    StructField("ss_ext_list_price", MONTANT),
    StructField("ss_ext_tax", MONTANT),
    StructField("ss_coupon_amt", MONTANT),
    StructField("ss_net_paid", MONTANT),
    StructField("ss_net_paid_inc_tax", MONTANT),
    StructField("ss_net_profit", MONTANT),
])
SCHEMA_PRODUITS = StructType([
    StructField("i_item_sk", LongType()),
    StructField("i_item_id", StringType()),
    StructField("i_rec_start_date", DateType()),
    StructField("i_rec_end_date", DateType()),
    StructField("i_item_desc", StringType()),
    StructField("i_current_price", MONTANT),
    StructField("i_wholesale_cost", MONTANT),
    StructField("i_brand_id", LongType()),
    StructField("i_brand", StringType()),
    StructField("i_class_id", IntegerType()),
    StructField("i_class", StringType()),
    StructField("i_category_id", IntegerType()),
    StructField("i_category", StringType()),
    StructField("i_manufact_id", LongType()),
    StructField("i_manufact", StringType()),
    StructField("i_size", StringType()),
    StructField("i_formulation", StringType()),
    StructField("i_color", StringType()),
    StructField("i_units", StringType()),
    StructField("i_container", StringType()),
    StructField("i_manager_id", LongType()),
    StructField("i_product_name", StringType()),
])

def lire(chemin, schema):
    lecture = spark.read.options(**LECTURE).schema(schema)
    return lecture.csv("file://" + chemin)

def lignes_fichier(chemin):
    total = 0
    with open(chemin, "rb") as fichier:
        for _ in fichier:
            total += 1
    return total

def octets_hdfs(chemin):
    sortie = subprocess.check_output(["hdfs", "dfs", "-du", "-s", chemin], text=True).strip()
    return int(sortie.split()[0]), sortie

print("Python", sys.version.split()[0])
print("Spark", spark.version)
print("mode", spark.sparkContext.master)
print("defaultFS", spark.conf.get("spark.hadoop.fs.defaultFS"))
print("entrées", VENTES, PRODUITS)
print("sorties", SILVER_VENTES, SILVER_PRODUITS, GOLD)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/19 10:12:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python 3.12.3
Spark 4.3.0
mode local[*]
defaultFS hdfs://localhost:9000
entrées /data/retail_project/data/store_sales/store_sales.dat /data/retail_project/data/item/item.dat
sorties /retail-lake/silver/uc06/store_sales /retail-lake/silver/uc06/item /retail-lake/gold/uc06/ventes_magasin_par_produit


## Règles de nettoyage

| Anomalie | Règle | Action | Justification |
| --- | --- | --- | --- |
| Pas d'en-tête, tout est texte | Largeur 23 pour les ventes, 22 pour les produits | Schéma explicite, sinon arrêt | Noms de la spécification TPCx-BB, posés seulement si la largeur observée correspond |
| Champ vide | Champ vide lu comme null | Conservé s'il est facultatif | Les clés secondaires (foyer, adresse, promotion) peuvent être absentes |
| Identifiant de ligne absent | Date, article, magasin, ticket, quantité et montant net requis | Ligne écartée et comptée | Sans ces champs, la ligne n'est pas une vente exploitable |
| Quantité nulle ou négative | `ss_quantity` doit être > 0 | Ligne écartée et comptée | Une ligne de ticket sans quantité positive n'entre pas dans le chiffre d'affaires |
| Doublon exact | Ligne identique sur toutes les colonnes | Supprimé et compté | La définition est la ligne entière, pas le ticket |
| Profit négatif | `ss_net_profit` peut être < 0 | Conservé | Un profit de ligne négatif n'est pas une erreur de fichier |
| Fin de version produit vide | `i_rec_end_date` null | Conservé | Aucune fin de version n'est observée, ce n'est pas un article invalide |
| Texte mélangé à des nombres | Marque, fabricant, formulation, nom | Laissés en texte | Un nombre isolé dans un libellé ne change pas le type de la colonne |


In [2]:
ventes_brutes = lire(VENTES, SCHEMA_VENTES)
produits_bruts = lire(PRODUITS, SCHEMA_PRODUITS)

print("ventes : lignes fichier", lignes_fichier(VENTES), "lignes lues", ventes_brutes.count())
print("produits : lignes fichier", lignes_fichier(PRODUITS), "lignes lues", produits_bruts.count())
ventes_brutes.printSchema()
produits_bruts.printSchema()

clefs_vente = ["ss_sold_date_sk", "ss_item_sk", "ss_store_sk", "ss_ticket_number", "ss_quantity", "ss_net_paid"]
masque_ecart = F.col("ss_quantity").isNull() | (F.col("ss_quantity") <= 0)
for clef in clefs_vente:
    if clef != "ss_quantity":
        masque_ecart = masque_ecart | F.col(clef).isNull()

ventes_ecartees = ventes_brutes.filter(masque_ecart)
ventes = ventes_brutes.filter(~masque_ecart)
nb_avant = ventes.count()
ventes = ventes.dropDuplicates()
nb_doublons = nb_avant - ventes.count()
print("ventes écartées", ventes_ecartees.count())
print("doublons exacts retirés", nb_doublons)
print("profits négatifs conservés", ventes.filter(F.col("ss_net_profit") < 0).count())

produits_ecartes = produits_bruts.filter(F.col("i_item_sk").isNull() | F.col("i_item_id").isNull())
produits = produits_bruts.filter(F.col("i_item_sk").isNotNull() & F.col("i_item_id").isNotNull())
nb_avant_produits = produits.count()
produits = produits.dropDuplicates(["i_item_sk"])
print("produits écartés", produits_ecartes.count())
print("i_item_sk en double retirés", nb_avant_produits - produits.count())
print("i_rec_end_date vides", produits.filter(F.col("i_rec_end_date").isNull()).count())

ventes = (
    ventes
    .withColumn("source_fichier", F.lit(VENTES))
    .withColumn("source_lot", F.lit("initial"))
)
produits = (
    produits
    .withColumn("source_fichier", F.lit(PRODUITS))
    .withColumn("source_lot", F.lit("initial"))
)
ventes = ventes.persist()
produits = produits.persist()
print("silver ventes", ventes.count(), "silver produits", produits.count())


ventes : lignes fichier 667579 lignes lues 667579


produits : lignes fichier 17820 lignes lues 17820
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable =

ventes écartées 0
doublons exacts retirés 0


profits négatifs conservés 495971


produits écartés 0


i_item_sk en double retirés 0


i_rec_end_date vides 17820


silver ventes 667579 silver produits 17820


## Jointure

Relation vérifiée au Jour 1 : `ss_item_sk` vers `i_item_sk`. Jointure gauche pour conserver chaque ligne de vente. Si le nombre de lignes augmente, le référentiel produit n'est pas unique et le résultat n'est pas écrit.


In [3]:
referentiel = produits.select(
    "i_item_sk", "i_item_id", "i_product_name", "i_category", "i_class"
)
lignes_avant = ventes.count()
cles_produit = produits.select("i_item_sk").distinct().count()
print("clés produit distinctes", cles_produit, "lignes produit", produits.count())

enrichi = ventes.join(referentiel, ventes.ss_item_sk == referentiel.i_item_sk, "left")
lignes_apres = enrichi.count()
orphelins = enrichi.filter(F.col("i_item_id").isNull()).count()
print("lignes avant", lignes_avant, "lignes après", lignes_apres, "orphelins", orphelins)
if lignes_apres != lignes_avant:
    raise SystemExit("la jointure multiplie les lignes, écriture arrêtée")
enrichi = enrichi.persist()
enrichi.select(
    "ss_ticket_number", "ss_item_sk", "i_item_id", "i_category", "ss_quantity", "ss_net_paid"
).show(5, truncate=False)


clés produit distinctes 17820 lignes produit 17820


lignes avant 667579 lignes après 667579 orphelins 0


+----------------+----------+----------------+------------------------+-----------+-----------+
|ss_ticket_number|ss_item_sk|i_item_id       |i_category              |ss_quantity|ss_net_paid|
+----------------+----------+----------------+------------------------+-----------+-----------+
|32              |12638     |AAAAAAAAAAAAASSC|Tools & Home Improvement|65         |162.50     |
|141             |2571      |AAAAAAAAAAAAADUX|Movies & TV             |28         |1.45       |
|237             |1630      |AAAAAAAAAAAAACKS|Movies & TV             |2          |32.50      |
|303             |14435     |AAAAAAAAAAAAAVJF|Music                   |79         |17.38      |
|340             |12673     |AAAAAAAAAAAAASTL|Books                   |63         |190.26     |
+----------------+----------+----------------+------------------------+-----------+-----------+
only showing top 5 rows


## Gold – ventes magasin par produit

| Élément | Description |
| --- | --- |
| Nom | `ventes_magasin_par_produit` |
| Objectif | Suivre le chiffre d'affaires net des ventes magasin par produit |
| Sources | Silver `store_sales` et Silver `item`, lot initial |
| Granularité | Une ligne par `i_item_sk` |
| Mesures | `nb_lignes`, `quantite`, `ca_net` (somme de `ss_net_paid`), `ca_net_ttc`, `profit_net` |
| Dimensions | Identifiant, nom, catégorie, classe |
| Limite | Lot initial seulement. Les montants restent dans l'unité du fichier. Le profit négatif de certaines lignes est inclus. |


In [4]:
gold = (
    enrichi.groupBy("i_item_sk", "i_item_id", "i_product_name", "i_category", "i_class")
    .agg(
        F.count("*").alias("nb_lignes"),
        F.sum("ss_quantity").alias("quantite"),
        F.sum("ss_net_paid").alias("ca_net"),
        F.sum("ss_net_paid_inc_tax").alias("ca_net_ttc"),
        F.sum("ss_net_profit").alias("profit_net"),
    )
)
print("produits dans le gold", gold.count())
print("lignes de vente couvertes", gold.agg(F.sum("nb_lignes")).collect()[0][0])
print("ca_net total", gold.agg(F.sum("ca_net")).collect()[0][0])
gold.orderBy(F.col("ca_net").desc()).show(10, truncate=False)


produits dans le gold 17820


lignes de vente couvertes 667579


ca_net total 1150529586.37


+---------+----------------+-------------------------------------------------+----------------------+--------------------------+---------+--------+-----------+-----------+-----------+
|i_item_sk|i_item_id       |i_product_name                                   |i_category            |i_class                   |nb_lignes|quantite|ca_net     |ca_net_ttc |profit_net |
+---------+----------------+-------------------------------------------------+----------------------+--------------------------+---------+--------+-----------+-----------+-----------+
|10000    |AAAAAAAAAAAAAOUQ|vwPR6l3UPyCIv5bljZbIG9k9YHLajogyqKbd6            |Home & Kitchen        |Kitchen Utensils & Gadgets|9029     |454954  |15408393.55|16108841.77|-7521122.26|
|10003    |AAAAAAAAAAAAAOUT|Z2kc                                             |Electronics           |Computer Accessories      |8761     |439086  |14933870.10|15614283.97|-7411910.29|
|10001    |AAAAAAAAAAAAAOUR|6VxbaI6thqY49zO                                  |Mu

## Écriture Parquet

Parquet stocke les colonnes avec leur type. Ce n'est pas une table transactionnelle : il n'y a ni transaction ACID, ni voyage dans le temps. On relit ensuite le schéma, un échantillon et le nombre de lignes.


In [5]:
(
    ventes.write.mode("overwrite")
    .option("compression", "snappy")
    .parquet(SILVER_VENTES)
)
(
    produits.write.mode("overwrite")
    .option("compression", "snappy")
    .parquet(SILVER_PRODUITS)
)
(
    gold.write.mode("overwrite")
    .option("compression", "snappy")
    .parquet(GOLD)
)

for libelle, chemin, attendu in (
    ("silver ventes", SILVER_VENTES, ventes.count()),
    ("silver produits", SILVER_PRODUITS, produits.count()),
    ("gold", GOLD, gold.count()),
):
    relu = spark.read.parquet(chemin)
    print("====", libelle, chemin)
    relu.printSchema()
    print("lignes relues", relu.count(), "lignes attendues", attendu)
    relu.show(3, truncate=False)


==== silver ventes /retail-lake/silver/uc06/store_sales
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (null

lignes relues 667579 lignes attendues 667579


+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+-----------------------------------------------------+----------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|source_fichier                                       |source_lot|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+-

lignes relues 17820 lignes attendues 17820


+---------+----------------+----------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------------+----------+-----------------------------------------------+----------+-----------------------+-------------+-----------+-------------+-------------------------------------+-------+------------------+-------+-------+-----------+------------+---------------------------------------------+---------------------------------------+----------+
|i_item_sk|i_item_id       |i_rec_start_date|i_rec_end_date|i_item_desc                                                                                                                                                                                        |i_current_price|i_wholesale_cost|i_brand_id|i_brand                                        |i_class_id|i_class       

lignes relues 17820 lignes attendues 17820
+---------+----------------+-------------------------------------------+------------+--------------------+---------+--------+--------+----------+----------+
|i_item_sk|i_item_id       |i_product_name                             |i_category  |i_class             |nb_lignes|quantite|ca_net  |ca_net_ttc|profit_net|
+---------+----------------+-------------------------------------------+------------+--------------------+---------+--------+--------+----------+----------+
|6195     |AAAAAAAAAAAAAJEH|abCS7lpxxv6szEOTMUZwulcjB4HOOGmeP2ixlRrKoif|Books       |New                 |27       |1094    |32375.05|34271.16  |-25437.22 |
|10010    |AAAAAAAAAAAAAOVA|jMHE6kXJIOIJbZqWT9cfE2sMI5UHn8             |Toys & Games|Action & Toy Figures|19       |909     |33282.10|34648.53  |-8235.39  |
|13201    |AAAAAAAAAAAAATNT|pUS                                        |Music       |Rock                |21       |1160    |55807.15|58124.08  |-17428.90 |
+---------+----

## Comparaison des formats

Le source retenu est un fichier plat séparé par `|`, proche d'un CSV sans en-tête. JSON n'est pas présent dans ces deux sources. La comparaison de volume oppose le fichier plat des ventes au Parquet Silver, puis le Gold Parquet à un CSV écrit uniquement pour la mesure et effacé ensuite.


In [6]:
def hdfs(*args):
    return subprocess.check_output(["hdfs", "dfs", *args], text=True).strip()

gold.write.mode("overwrite").option("header", True).csv(CSV_COMPARAISON)
print("source ventes, plat, octets", os.path.getsize(VENTES))
print("silver ventes, parquet, octets", octets_hdfs(SILVER_VENTES)[0])
print("silver produits, parquet, octets", octets_hdfs(SILVER_PRODUITS)[0])
print("gold parquet, octets", octets_hdfs(GOLD)[0])
print("gold csv de comparaison, octets", octets_hdfs(CSV_COMPARAISON)[0])
print(hdfs("-ls", "/retail-lake/silver/uc06"))
print(hdfs("-ls", "/retail-lake/gold/uc06"))
subprocess.check_call(["hdfs", "dfs", "-rm", "-r", "-f", "/retail-lake/tmp/uc06"])
print("csv de comparaison supprimé")


source ventes, plat, octets 92764700


silver ventes, parquet, octets 57010955


silver produits, parquet, octets 5785633


gold parquet, octets 1147473


gold csv de comparaison, octets 1883641


Found 2 items
drwxr-xr-x   - etudiant supergroup          0 2026-11-19 10:14 /retail-lake/silver/uc06/item
drwxr-xr-x   - etudiant supergroup          0 2026-11-19 10:14 /retail-lake/silver/uc06/store_sales


Found 1 items
drwxr-xr-x   - etudiant supergroup          0 2026-11-19 10:14 /retail-lake/gold/uc06/ventes_magasin_par_produit


Deleted /retail-lake/tmp/uc06


csv de comparaison supprimé


## Contrôles qualité


In [7]:
relu_ventes = spark.read.parquet(SILVER_VENTES)
relu_produits = spark.read.parquet(SILVER_PRODUITS)
relu_gold = spark.read.parquet(GOLD)

controles = []

def ajouter(nom, regle, valeur, ok):
    controles.append((nom, regle, str(valeur), "OK" if ok else "ECART"))

ajouter(
    "Volume ventes",
    "Lignes Silver = lignes du fichier, si rien n'est écarté",
    f"{relu_ventes.count()} / {lignes_fichier(VENTES)}",
    relu_ventes.count() == lignes_fichier(VENTES),
)
ajouter(
    "Volume produits",
    "Lignes Silver = lignes du fichier, si rien n'est écarté",
    f"{relu_produits.count()} / {lignes_fichier(PRODUITS)}",
    relu_produits.count() == lignes_fichier(PRODUITS),
)
ajouter(
    "Complétude vente",
    "Date, article, magasin, ticket, quantité et montant net non nuls",
    ventes_ecartees.count(),
    ventes_ecartees.count() == 0,
)
ajouter(
    "Unicité produit",
    "i_item_sk unique dans Silver item",
    relu_produits.count() - relu_produits.select("i_item_sk").distinct().count(),
    relu_produits.count() == relu_produits.select("i_item_sk").distinct().count(),
)
ajouter(
    "Validité quantité",
    "ss_quantity > 0",
    relu_ventes.filter(F.col("ss_quantity") <= 0).count(),
    relu_ventes.filter(F.col("ss_quantity") <= 0).count() == 0,
)
ajouter(
    "Validité montant",
    "ss_net_paid non nul",
    relu_ventes.filter(F.col("ss_net_paid").isNull()).count(),
    relu_ventes.filter(F.col("ss_net_paid").isNull()).count() == 0,
)
ajouter(
    "Intégrité article",
    "Chaque ss_item_sk trouve un i_item_sk",
    orphelins,
    orphelins == 0 and lignes_apres == lignes_avant,
)
ajouter(
    "Cohérence gold",
    "La somme des lignes Gold égale les lignes de vente jointes",
    relu_gold.agg(F.sum("nb_lignes")).collect()[0][0],
    relu_gold.agg(F.sum("nb_lignes")).collect()[0][0] == lignes_apres,
)
ajouter(
    "Cohérence profit",
    "Les profits négatifs sont conservés, pas corrigés",
    relu_ventes.filter(F.col("ss_net_profit") < 0).count(),
    True,
)

bilan = spark.createDataFrame(controles, ["controle", "regle", "resultat", "statut"])
bilan.show(20, truncate=False)
print("écarts", bilan.filter(F.col("statut") == "ECART").count())


+-----------------+----------------------------------------------------------------+---------------+------+
|controle         |regle                                                           |resultat       |statut|
+-----------------+----------------------------------------------------------------+---------------+------+
|Volume ventes    |Lignes Silver = lignes du fichier, si rien n'est écarté         |667579 / 667579|OK    |
|Volume produits  |Lignes Silver = lignes du fichier, si rien n'est écarté         |17820 / 17820  |OK    |
|Complétude vente |Date, article, magasin, ticket, quantité et montant net non nuls|0              |OK    |
|Unicité produit  |i_item_sk unique dans Silver item                               |0              |OK    |
|Validité quantité|ss_quantity > 0                                                 |0              |OK    |
|Validité montant |ss_net_paid non nul                                             |0              |OK    |
|Intégrité article|Chaque ss

écarts 0


## Reproduction

1. Démarrer le conteneur `lake`. HDFS et Jupyter doivent répondre.
2. Ouvrir `uc06_exploration.ipynb`, exécuter toutes les cellules. Il ne doit rien écrire.
3. Ouvrir ce notebook et exécuter toutes les cellules.
4. Les résultats sont dans `/retail-lake/silver/uc06` et `/retail-lake/gold/uc06`.

Versions, mode `local[*]` et chemins sont imprimés par la première cellule. Le traitement couvre le lot initial seulement. Les noms de colonnes sont ceux de la spécification TPCx-BB. Parquet ne remplace pas une table transactionnelle.
